# Exercises: Data Fundamentals

Practice core NumPy, Pandas, Matplotlib, and data-wrangling skills.
Each exercise is self-contained — run the cells in order.

## Exercise 1: Array Broadcasting Challenge

Given a matrix of student scores (rows = students, cols = subjects) and a 1-D array of subject weights, compute **weighted averages** per student using broadcasting — no Python loops allowed.

**Requirements:**
- Create a 6×4 random score matrix (values 50-100)
- Weights: `[0.3, 0.25, 0.25, 0.2]`
- Normalise weights so they sum to 1
- Compute weighted average per student via broadcasting
- Find the top-performing student index

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np

np.random.seed(42)
scores = np.random.randint(50, 101, size=(6, 4))
weights = np.array([0.3, 0.25, 0.25, 0.2])
weights = weights / weights.sum()  # normalise

# Broadcasting: (6,4) * (4,) -> (6,4), then sum across columns
weighted_avg = (scores * weights).sum(axis=1)

top_student = np.argmax(weighted_avg)
print("Scores matrix:\n", scores)
print("\nWeighted averages:", np.round(weighted_avg, 2))
print(f"Top student: index {top_student} with avg {weighted_avg[top_student]:.2f}")


### Explanation

NumPy broadcasts the 1-D weight vector across every row of the 2-D score matrix, performing element-wise multiplication without an explicit loop. Summing along `axis=1` collapses the subject dimension, yielding one weighted average per student.

## Exercise 2: Pandas GroupBy + Merge Puzzle

You have two DataFrames — `orders` and `customers`. Merge them and answer:
1. Total revenue per city
2. Average order value per customer tier (Gold/Silver/Bronze)
3. Top 3 customers by total spend

**Requirements:**
- Build both DataFrames from synthetic data
- Use `pd.merge`, `groupby`, and `agg`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

customers = pd.DataFrame({
    'customer_id': range(1, 21),
    'name': [f'Customer_{i}' for i in range(1, 21)],
    'city': np.random.choice(['New York', 'Chicago', 'LA', 'Houston'], 20),
    'tier': np.random.choice(['Gold', 'Silver', 'Bronze'], 20),
})

orders = pd.DataFrame({
    'order_id': range(1, 101),
    'customer_id': np.random.choice(range(1, 21), 100),
    'amount': np.round(np.random.exponential(80, 100), 2),
})

merged = pd.merge(orders, customers, on='customer_id')

# 1. Total revenue per city
rev_by_city = merged.groupby('city')['amount'].sum().sort_values(ascending=False)
print("Revenue by city:\n", rev_by_city)

# 2. Avg order value per tier
avg_by_tier = merged.groupby('tier')['amount'].mean()
print("\nAvg order value by tier:\n", avg_by_tier.round(2))

# 3. Top 3 customers
top3 = (merged.groupby(['customer_id', 'name'])['amount']
        .sum().sort_values(ascending=False).head(3))
print("\nTop 3 customers:\n", top3)


### Explanation

`pd.merge` performs an inner join on `customer_id`. `groupby().agg()` then splits the merged frame by the desired categorical column and computes aggregates in a single vectorised pass.

## Exercise 3: Custom Visualisation

Create a **2×2 subplot figure** with:
1. Scatter plot with colour-coded clusters
2. Histogram with KDE overlay
3. Box plot comparing groups
4. Heatmap of a correlation matrix

**Requirements:**
- Use `matplotlib` + `seaborn`
- Proper titles, axis labels, and a shared `fig.suptitle`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)

# Synthetic data
n = 200
clusters = np.random.choice([0, 1, 2], n)
x = np.random.randn(n) + clusters * 3
y = np.random.randn(n) + clusters * 2
values = np.random.exponential(5, n)
groups = np.random.choice(['A', 'B', 'C'], n)
df = pd.DataFrame({'x': x, 'y': y, 'cluster': clusters,
                    'value': values, 'group': groups,
                    'feat1': np.random.randn(n),
                    'feat2': np.random.randn(n) * 2,
                    'feat3': np.random.randn(n) + 1})

fig, axes = plt.subplots(2, 2, figsize=(12, 10))
fig.suptitle('Data Fundamentals — Custom Visualisation', fontsize=14)

# 1 Scatter
scatter = axes[0, 0].scatter(x, y, c=clusters, cmap='viridis', alpha=0.6, s=30)
axes[0, 0].set_title('Clustered Scatter')
axes[0, 0].set_xlabel('X'); axes[0, 0].set_ylabel('Y')
plt.colorbar(scatter, ax=axes[0, 0], label='Cluster')

# 2 Histogram + KDE
sns.histplot(values, kde=True, ax=axes[0, 1], color='steelblue')
axes[0, 1].set_title('Distribution of Values')
axes[0, 1].set_xlabel('Value')

# 3 Box plot
sns.boxplot(data=df, x='group', y='value', ax=axes[1, 0], palette='Set2')
axes[1, 0].set_title('Value by Group')

# 4 Correlation heatmap
corr = df[['x', 'y', 'value', 'feat1', 'feat2', 'feat3']].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', ax=axes[1, 1])
axes[1, 1].set_title('Correlation Matrix')

plt.tight_layout()
plt.savefig('exercises/ex1_visualization.png', dpi=100)
plt.close()
print("Figure saved to exercises/ex1_visualization.png")


### Explanation

`plt.subplots(2,2)` creates a grid of axes. Each subplot uses either raw matplotlib (scatter + colorbar) or seaborn helpers (`histplot`, `boxplot`, `heatmap`). `tight_layout()` prevents label overlap.

## Exercise 4: Time Series Resampling

Simulate 1 year of daily sales data (with weekly seasonality and trend). Then:
1. Resample to weekly and monthly totals
2. Compute a 7-day rolling mean
3. Identify the month with highest sales

**Requirements:**
- `pd.date_range` with freq='D'
- Use `.resample()` and `.rolling()`

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)
dates = pd.date_range('2023-01-01', periods=365, freq='D')
trend = np.linspace(100, 200, 365)
seasonality = 20 * np.sin(2 * np.pi * np.arange(365) / 7)
noise = np.random.normal(0, 10, 365)
sales = trend + seasonality + noise

ts = pd.Series(sales, index=dates, name='daily_sales')

weekly = ts.resample('W').sum()
monthly = ts.resample('ME').sum()
rolling_7 = ts.rolling(7).mean()

best_month = monthly.idxmax().strftime('%B %Y')

print("Weekly sales (first 5 weeks):\n", weekly.head())
print("\nMonthly sales:\n", monthly)
print(f"\nBest month: {best_month} — ${monthly.max():,.0f}")
print("\n7-day rolling mean (last 5):\n", rolling_7.tail())


### Explanation

`.resample('W')` groups by ISO week, `.resample('ME')` by calendar month-end. `.rolling(7).mean()` gives a centred-by-default moving average that smooths out daily noise while preserving the underlying trend.

## Exercise 5: Data Cleaning Pipeline

You receive a messy DataFrame with:
- Missing values in multiple columns
- Duplicate rows
- Inconsistent string formatting
- Outliers in a numeric column

Build a cleaning pipeline (a single function) that returns a tidy DataFrame.

**Requirements:**
- Handle each issue explicitly
- Print a summary of what was cleaned

In [ ]:
# YOUR CODE HERE

### Solution

In [ ]:
import numpy as np
import pandas as pd

np.random.seed(42)

# Build messy data
raw = pd.DataFrame({
    'Name': ['Alice', 'bob', 'CHARLIE', 'Alice', 'Diana', 'bob',
             'Eve', None, 'Frank', 'Alice'],
    'Age': [25, 30, None, 25, 45, 30, 28, 35, 300, 25],   # 300 is an outlier
    'Salary': [50000, None, 70000, 50000, 90000, None,
               62000, 55000, 80000, 50000],
    'Department': [' Sales', 'Engineering', 'sales ', ' Sales', 'HR',
                   'engineering', 'HR', 'Sales', 'Engineering', ' Sales'],
})


def clean_dataframe(df):
    report = {}
    df = df.copy()

    # 1. Duplicates
    n_dup = df.duplicated().sum()
    df = df.drop_duplicates()
    report['duplicates_removed'] = int(n_dup)

    # 2. String normalisation
    df['Name'] = df['Name'].str.strip().str.title()
    df['Department'] = df['Department'].str.strip().str.title()

    # 3. Missing values
    n_missing = df.isna().sum().sum()
    df['Name'] = df['Name'].fillna('Unknown')
    df['Age'] = df['Age'].fillna(df['Age'].median())
    df['Salary'] = df['Salary'].fillna(df['Salary'].median())
    report['missing_filled'] = int(n_missing)

    # 4. Outlier capping (IQR method)
    for col in ['Age', 'Salary']:
        q1, q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
        n_out = ((df[col] < lower) | (df[col] > upper)).sum()
        df[col] = df[col].clip(lower, upper)
        report[f'{col}_outliers_capped'] = int(n_out)

    print("Cleaning report:", report)
    return df


cleaned = clean_dataframe(raw)
print("\nCleaned DataFrame:")
print(cleaned.to_string(index=False))


### Explanation

The pipeline follows a deterministic order: remove duplicates → normalise strings → impute missing values → cap outliers. Using `clip` with IQR bounds is a robust, non-parametric way to handle outliers without removing rows.